In [65]:
import os

os.environ["PYSPARK_PYTHON"] = "python"
os.environ["PYSPARK_DRIVER_PYTHON"] = "python"

# STEP 1: SETUP PYSPARK JOB

## Basic Spark Setup

In [66]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[2]") \
    .appName("SalaryDetectionJob") \
    .config("spark.ui.enabled", "false") \
    .config("spark.sql.shuffle.partitions", "2") \
    .getOrCreate()

## Entry point

In [67]:
df = spark.read.csv(
    "C:/data/salary_dataset_9k.csv",
    header=True,
    inferSchema=True
).cache()

df.count()

9053

# STEP 2: CLEAN & STANDARDIZE DATA

In [68]:
from pyspark.sql.functions import to_date, col

df = df.withColumn(
    "Date",
    to_date(col("Date"), "dd-MM-yyyy")
)

print("After date parsing:", df.count())
df.select("Date").show(5, False)

df = df.filter(col("Date").isNotNull())
print("After filter:", df.count())

After date parsing: 9053
+----------+
|Date      |
+----------+
|2024-09-01|
|2024-09-01|
|2024-09-01|
|2024-09-01|
|2024-09-01|
+----------+
only showing top 5 rows

After filter: 9053


In [69]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, expr, sum as spark_sum

window_txn = Window.partitionBy("CustomerId").orderBy("Date")

df = df.withColumn(
    "prev_balance",
    lag("Balance").over(window_txn)
)

In [70]:
from pyspark.sql.functions import upper, when

salary_txn_df = df.filter(
    upper(col("Transaction Details")).rlike("SAL|SALARY|PAYROLL")
)

post_salary_spend = df.join(
    salary_txn_df.select("CustomerId", "Date").withColumnRenamed("Date", "salary_date"),
    "CustomerId"
).filter(
    (col("Date") > col("salary_date")) &
    (col("Date") <= col("salary_date") + expr("INTERVAL 3 DAYS"))
)

behavior_df = post_salary_spend.groupBy("CustomerId").agg(
    spark_sum(when(col("Type") == "DEBIT", col("Amount")).otherwise(0)).alias("spend_after_salary")
)
behavior_df = behavior_df.join(
    df.groupBy("CustomerId").agg({"Balance": "min"}).withColumnRenamed("min(Balance)", "min_balance"),
    "CustomerId",
    "left"
)

behavior_df = behavior_df.withColumn(
    "quick_drain_flag",
    col("min_balance") < 2000
)

In [71]:
from pyspark.sql.functions import lag, col, upper
from pyspark.sql.window import Window

window_salary = Window.partitionBy("CustomerId").orderBy("Date")

salary_df = df.filter(
    upper(col("Transaction Details")).rlike("SAL|SALARY|PAYROLL|WAGE")
)

salary_df = salary_df.withColumn(
    "prev_salary",
    lag("Amount").over(window_salary)
)

salary_df = salary_df.withColumn(
    "salary_drop_flag",
    col("Amount") < 0.7 * col("prev_salary")
)

In [72]:
salary_drop_df = salary_df.groupBy("CustomerId").agg(
    spark_sum(when(col("salary_drop_flag"), 1).otherwise(0)).alias("salary_drop_count")
)

behavior_df = behavior_df.join(
    salary_drop_df,
    "CustomerId",
    "left"
)

# STEP 3: FILTER CREDIT TRANSACTIONS

## PySpark version:

In [73]:
print("df count:", df.count())

df.select("CustomerId").show(5)
df.select("Type").distinct().show()
df.select("Transaction Details").show(5, False)

df count: 9053
+----------+
|CustomerId|
+----------+
|        C1|
|        C1|
|        C1|
|        C1|
|        C1|
+----------+
only showing top 5 rows

+------+
|  Type|
+------+
|CREDIT|
| DEBIT|
+------+

+-------------------+
|Transaction Details|
+-------------------+
|UPI/General Txn    |
|UPI/General Txn    |
|UPI/General Txn    |
|UPI/General Txn    |
|UPI/General Txn    |
+-------------------+
only showing top 5 rows



In [74]:
from pyspark.sql.functions import upper

credit_df = df.filter(
    (col("Type").isin("CREDIT", "CR")) &
    (col("Amount") >= 3000)
)
print("credit_df:", credit_df.count())
print("After filter:", df.count())
print("salary_txn_df:", salary_txn_df.count())
print("credit_df:", credit_df.count())


credit_df: 1931
After filter: 9053
salary_txn_df: 100
credit_df: 1931


In [75]:
all_customers_df = df.select("CustomerId").distinct()

# STEP 4: SENDER EXTRACTION

In [76]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

def extract_sender(details):
    if not details:
        return "UNKNOWN"

    details = details.upper()
    parts = details.split("/")

    if len(parts) >= 2:
        sender = parts[1]
        words = sender.split()

        sender = " ".join(words[:2])

        if sender.replace(" ", "").isdigit():
            return f"ANON_{sender.replace(' ', '')}"

        return sender.strip()

    return "UNKNOWN"

extract_sender_udf = udf(extract_sender, StringType())

In [77]:
credit_df = credit_df.withColumn(
    "sender",
    extract_sender_udf(col("Transaction Details"))
)


In [78]:
noise_patterns = "GENERAL|UPI|IMPS|NEFT|TRANSFER|PAYMENT"

credit_df = credit_df.filter(
    ~col("sender").rlike(noise_patterns)
)

# STEP 5: MERGE SAME-DAY TRANSACTIONS

In [79]:
from pyspark.sql.functions import sum as spark_sum

daily_df = credit_df.groupBy(
    "CustomerId", "sender", "Date"
).agg(
    spark_sum("Amount").alias("daily_amount")
)

# STEP 6: COMPUTE INTERVALS

In [80]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, datediff
from pyspark.sql.functions import collect_list
window_spec = Window.partitionBy("CustomerId", "sender").orderBy("Date")

daily_df = daily_df.withColumn(
    "prev_date",
    lag("Date").over(window_spec)
)

daily_df = daily_df.withColumn(
    "interval_days",
    datediff(col("Date"), col("prev_date"))
)
print("daily_df:", daily_df.count())

daily_df: 43


# STEP 7: AGGREGATE FEATURES

## Feature aggregation:

In [81]:
from pyspark.sql.functions import collect_list, avg, stddev, count

features_df = daily_df.groupBy(
    "CustomerId", "sender"
).agg(
    avg("interval_days").alias("interval_mean"),
    stddev("interval_days").alias("interval_std"),
    avg("daily_amount").alias("amount_mean"),
    stddev("daily_amount").alias("amount_std"),
    count("*").alias("count"),
    collect_list("daily_amount").alias("salary_history")
)

In [82]:
from pyspark.sql.functions import when

features_df = features_df.withColumn(
    "salary_pattern",
    when(col("interval_mean").between(25, 35), "MONTHLY")
    .when(col("interval_mean").between(12, 18), "BI-WEEKLY")
    .when(col("interval_mean").between(5, 9), "WEEKLY")
    .when(col("interval_mean") > 40, "IRREGULAR_GAP")
    .otherwise("IRREGULAR")
)
print("features_df:", features_df.count())


features_df: 18


In [83]:
from pyspark.sql.functions import col, lower

# 1. periodic check
features_df = features_df.withColumn(
    "is_periodic",
    (
        col("interval_mean").isNotNull() &
        (
            col("interval_mean").between(24, 35) |   # monthly
            col("interval_mean").between(5, 10)  |   # weekly
            col("interval_mean").between(10, 22)     # semi-monthly / flexible
        )
    )
)

# 2. stable amount
features_df = features_df.withColumn(
    "is_stable_amount",
    col("amount_std") < 10000
)

# 3. repetition
features_df = features_df.withColumn(
    "has_repetition",
    col("count") >= 2
)

features_df = features_df.withColumn(
    "has_salary_keyword",
    col("sender").rlike("SALARY|PAYROLL|SAL|WAGE|COMPANY")
)

features_df = features_df.withColumn(
    "fnf_flag",
    (col("amount_mean") > 100000) &
    (col("interval_mean") > 45)
)
features_df = features_df.withColumn(
    "bonus_flag",
    col("amount_std") > (0.5 * col("amount_mean"))
)
features_df = features_df.withColumn(
    "fnf_flag",
    (col("amount_mean") > 80000) &
    (col("interval_mean") > 45)
)

In [84]:
from pyspark.sql.functions import when

features_df = features_df.withColumn(
    "salary_confidence",
    when(col("count") >= 3, "HIGH")
    .when(col("count") == 2, "MEDIUM")
    .otherwise("LOW")
)

In [85]:
features_df = features_df.withColumn(
    "is_salary_account",
    (
        (
            col("is_periodic") &
            col("has_repetition")
        ) &
        (
            col("has_salary_keyword") |
            (col("amount_mean") > 20000)
        )
    )
)

In [86]:
from pyspark.sql.functions import sort_array

features_df = features_df.withColumn(
    "salary_history",
    sort_array(col("salary_history"))
)

In [87]:
from pyspark.sql.functions import coalesce, lit

features_df = features_df.withColumn(
    "interval_std",
    coalesce(col("interval_std"), lit(0.0))
)

features_df = features_df.withColumn(
    "amount_std",
    coalesce(col("amount_std"), lit(0.0))
)

features_df = features_df.withColumn(
    "amount_mean",
    coalesce(col("amount_mean"), lit(1.0))
)

# STEP 8: APPLY SALARY RULES

In [88]:
features_df = features_df.withColumn(
    "is_periodic",
    (
        col("interval_mean").isNotNull() &
        (
            col("interval_mean").between(24, 35) |   # monthly
            col("interval_mean").between(5, 10)  |   # weekly
            col("interval_mean").between(10, 22)     # semi-monthly / flexible
        )
    )
)

features_df = features_df.withColumn(
    "is_stable_amount",
    col("amount_std") < 10000
)

features_df = features_df.withColumn(
    "has_repetition",
    col("count") >= 2
)


In [89]:
from pyspark.sql.functions import max as spark_max, datediff

# last salary date per customer
last_salary_df = salary_df.groupBy("CustomerId").agg(
    spark_max("Date").alias("last_salary_date")
)

# latest transaction date per customer
last_txn_df = df.groupBy("CustomerId").agg(
    spark_max("Date").alias("last_txn_date")
)

salary_status_df = last_salary_df.join(
    last_txn_df, "CustomerId"
)

salary_status_df = salary_status_df.withColumn(
    "days_since_last_salary",
    datediff(col("last_txn_date"), col("last_salary_date"))
)

In [90]:
salary_status_df = salary_status_df.withColumn(
    "salary_missed_flag",
    col("days_since_last_salary") > 40
)

In [91]:
df.filter(col("CustomerId") == "C4").show(truncate=False)

+----------+----------+-------------------+------+-------+--------+--------+------------+
|CustomerId|Date      |Transaction Details|Type  |Amount |Balance |Category|prev_balance|
+----------+----------+-------------------+------+-------+--------+--------+------------+
|C4        |2024-09-01|UPI/General Txn    |DEBIT |2070.13|11032.44|General |NULL        |
|C4        |2024-09-01|UPI/General Txn    |DEBIT |2511.03|53380.56|General |11032.44    |
|C4        |2024-09-01|UPI/General Txn    |CREDIT|4502.7 |13260.42|General |53380.56    |
|C4        |2024-09-01|UPI/General Txn    |DEBIT |2488.32|77175.31|General |13260.42    |
|C4        |2024-09-01|UPI/General Txn    |CREDIT|1325.66|40140.52|General |77175.31    |
|C4        |2024-09-01|UPI/General Txn    |CREDIT|1566.5 |89327.15|General |40140.52    |
|C4        |2024-09-01|UPI/General Txn    |CREDIT|2728.49|11760.8 |General |89327.15    |
|C4        |2024-09-01|UPI/General Txn    |CREDIT|2249.0 |17021.6 |General |11760.8     |
|C4       

# STEP 9: SCORING

## Score calculation:

In [92]:
features_df = features_df.withColumn(
    "time_score",
    1 / (1 + col("interval_std"))
)

features_df = features_df.withColumn(
    "amount_score",
    1 / (1 + (col("amount_std") / (col("amount_mean") + 1)))
)

features_df = features_df.withColumn(
    "is_pattern_salary",
    (
        col("is_periodic") &
        col("has_repetition") &
        (col("amount_mean") > 20000) &
        (col("interval_std") < 10)
    )
)

features_df = features_df.withColumn(
    "final_score",
    (
        0.35 * col("time_score") +
        0.30 * col("amount_score") +
        0.20 * when(col("has_salary_keyword"), 1).otherwise(0) +
        0.15 * when(col("is_pattern_salary"), 1).otherwise(0)
    )
)

features_df = features_df.withColumn(
    "is_fake_salary",
    col("sender").rlike("FRIEND|SELF|OWN")
)

features_df = features_df.withColumn(
    "is_salary_account",
    (
        (col("is_salary_account") | col("is_pattern_salary")) &
        (~col("is_fake_salary"))
    )
)

# STEP 10: PICK BEST SENDER PER CUSTOMER

## Window ranking

In [93]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

rank_window = Window.partitionBy("CustomerId").orderBy(col("final_score").desc())

ranked_df = features_df.withColumn(
    "rank",
    row_number().over(rank_window)
)

best_df = ranked_df.filter(col("rank") == 1).select(
    "*",
    "is_pattern_salary",
    "is_fake_salary"
)

## STEP 11: OUTPUT

In [94]:
features_df.select(
    "CustomerId",
    "sender",
    "salary_pattern",
    "count",
    "interval_mean",
    "amount_mean",
    "is_periodic",
    "is_stable_amount",
    "has_repetition",
    "has_salary_keyword",
    "salary_confidence"
).show(truncate=False)

+----------+--------------+--------------+-----+-------------+------------------+-----------+----------------+--------------+------------------+-----------------+
|CustomerId|sender        |salary_pattern|count|interval_mean|amount_mean       |is_periodic|is_stable_amount|has_repetition|has_salary_keyword|salary_confidence|
+----------+--------------+--------------+-----+-------------+------------------+-----------+----------------+--------------+------------------+-----------------+
|C1        |INFOSYS SALARY|MONTHLY       |3    |30.5         |140000.0          |true       |false           |true          |true              |HIGH             |
|C1        |TCS SALARY    |MONTHLY       |3    |30.5         |160000.0          |true       |false           |true          |true              |HIGH             |
|C1        |WIPRO SALARY  |MONTHLY       |3    |30.5         |100000.0          |true       |false           |true          |true              |HIGH             |
|C10       |FRIEND SAL

In [95]:
# attach behavior FIRST
best_df = best_df.join(
    behavior_df,
    on="CustomerId",
    how="left"
)
# FIRST create final_df
final_df = all_customers_df.join(
    best_df,
    on="CustomerId",
    how="left"
)

# THEN join salary status
final_df = final_df.join(
    salary_status_df.select("CustomerId", "salary_missed_flag"),
    "CustomerId",
    "left"
)

In [96]:
from pyspark.sql.functions import col, when, lit, concat_ws, format_number, regexp_extract

final_output = final_df.select(
    col("CustomerId"),
    
    # sender
    when(
    col("sender").isNull() & (col("is_salary_account") == True),
    "PATTERN_BASED"
    ).when(
        col("sender").isNull(),
        "NO_SALARY_DETECTED"
    ).otherwise(col("sender"))
    .alias("sender"),

    # salary history (array → string)
    when(col("salary_history").isNull(), "-")
    .otherwise(concat_ws(", ", col("salary_history")))
    .alias("salary_history"),

    # avg salary (numeric safe)
    when(col("amount_mean").isNull(), "-")
    .otherwise(format_number(col("amount_mean"), 0))
    .alias("avg_salary"),

    # cycle
    when(col("salary_pattern").isNull(), "NO_PATTERN")
    .otherwise(col("salary_pattern"))
    .alias("salary_pattern"),

    when(col("salary_missed_flag") == True, "MISSED_SALARY")
    .when(col("salary_missed_flag").isNull(), "UNKNOWN")
    .otherwise("ACTIVE")
    .alias("salary_status"),

    # score
    when(col("final_score").isNull(), "0.00")
    .otherwise(format_number(col("final_score"), 2))
    .alias("score"),

    # confidence
    when(col("salary_confidence").isNull(), "NONE")
    .otherwise(col("salary_confidence"))
    .alias("salary_confidence"),

    # flag (boolean stays boolean)
    when(col("is_salary_account").isNull(), False)
    .otherwise(col("is_salary_account"))
    .alias("is_salary_account"),

    # case classification
    when(
        (col("is_salary_account") == True) &
        (
            col("has_salary_keyword") |   # strong signal
            (col("is_pattern_salary") == True)
        ),
        "POSITIVE"
    )
    .when(
            col("is_fake_salary") == True,
            "NEGATIVE_FAKE"
    )
    .when(
        col("sender") == "NO_SALARY_DETECTED",
        "NEGATIVE_NO_SALARY"
    )
    .when(
        col("salary_status") == "MISSED_SALARY",
        "NEGATIVE_MISSED"
    )
    .otherwise("NEGATIVE_WEAK_PATTERN")
    .alias("case_type"),

    # reason (boolean logic works properly now)
    when(col("sender").isNull(), "NO SALARY SIGNAL")
    .otherwise(
        concat_ws(" | ",
            when(col("is_periodic"), "Periodic").otherwise("Not Periodic"),
            when(col("is_stable_amount"), "Stable Amount").otherwise("Variable"),
            when(col("has_repetition"), "Repeated").otherwise("Not Repeated"),

            when(col("spend_after_salary").isNotNull(),
                concat_ws("", lit("Spend:"), col("spend_after_salary").cast("string"))
            ).otherwise("NoSpend"),

            when(col("quick_drain_flag") == True, "QuickDrain").otherwise("StableBalance"),
            when(col("salary_drop_count") > 0, "SalaryDrop").otherwise("NoDrop"),
            when(col("fnf_flag") == True, "FNF").otherwise("Regular"),
            when(col("has_salary_keyword"), "Keyword").otherwise("No Keyword")
        )
    ).alias("reason")
)
final_output = final_output.withColumn(
    "cust_num",
    regexp_extract(col("CustomerId"), "C(\\d+)", 1).cast("int")
).orderBy("cust_num").drop("cust_num")

final_output.show(truncate=False)

+----------+------------------+------------------------------------+----------+--------------+-------------+-----+-----------------+-----------------+---------------------+-----------------------------------------------------------------------------------------------------------+
|CustomerId|sender            |salary_history                      |avg_salary|salary_pattern|salary_status|score|salary_confidence|is_salary_account|case_type            |reason                                                                                                     |
+----------+------------------+------------------------------------+----------+--------------+-------------+-----+-----------------+-----------------+---------------------+-----------------------------------------------------------------------------------------------------------+
|C1        |INFOSYS SALARY    |100000.0, 140000.0, 180000.0        |140,000   |MONTHLY       |ACTIVE       |0.79 |HIGH             |true             |POSITIV